In [ ]:
import os
import sys
sys.path.append('../')
import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d
import torch
import seaborn as sns
import json


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
def visualize_point_cloud(xyz, rgb=None, save_path=None):
    """
    xyz: (N, 3) numpy array
    rgb: (N, 3) numpy array
    """
    point_cloud = o3d.geometry.PointCloud()
    point_cloud.points = o3d.utility.Vector3dVector(xyz)
    if rgb is not None:
        point_cloud.colors = o3d.utility.Vector3dVector(rgb)
    o3d.visualization.draw_geometries([point_cloud], )
    if save_path is not None:
        o3d.io.write_point_cloud(save_path, point_cloud)

## Show Initialized Point Cloud and Centers

In [ ]:
def vis_init_cano(data_dir, scene_name):
    with torch.no_grad():
        source_path = f'{data_dir}/{scene_name}'    
        pcd_path = f'{source_path}/point_cloud.ply'
        pcd = o3d.io.read_point_cloud(pcd_path)
        xyz, color = np.asarray(pcd.points), np.asarray(pcd.colors)
        
        joint_infos = json.load(open(f"{source_path}/joint_infos.json", "r"))
        center, scale, origin, direction = [], [], [], []
        K = len(joint_infos)
        for joint_info in joint_infos:
            center.append(joint_info['center'])
            scale.append(joint_info['dist_max'])
            if joint_info['joint_type'] == 'r':
                origin.append(joint_info['origin'])
            else:
                origin.append(joint_info['center'])
            direction.append(joint_info['direction'])
        center = np.array(center).reshape(K, 3)
        scale = np.array(scale).reshape(K, 1)
        origin = np.array(origin).reshape(K, 3)
        direction = np.array(direction).reshape(K, 3)
        num_slots = center.shape[0]
        pallete = np.array(sns.color_palette("hls", num_slots))
        pallete[0] = [0, 0, 0]

        track_data = np.load(f"{data_dir}/{scene_name}/{scene_name}_filtered_vis.npz")
        xyz = track_data["coords"][0]
        print(xyz.max(0), xyz.min(0))
        mask_ids = track_data["mask_ids"]
        color = pallete[mask_ids]

        # mannually correct the center
        
        # center[3] -= np.array([-0.2, +0.2, 0.])
        # center[4] -= np.array([0.2, 0.2, 0.])
        # center_info[:, :3] = center
        # # center_info[:, 3] /= 4
        # np.save(center_info_path, center_info)
        
        xyz_center = (center[None] + np.random.randn(1000, center.shape[0], 3) * 0.01).reshape(-1, 3)
        rgb_center = pallete[None].repeat(1000, 0).reshape(-1, 3)
        xyz_axis = origin[None] + direction[None] * np.linspace(0, np.ones_like(scale[None]) * 0.3, 100)[:, None]
        xyz_axis = xyz_axis.reshape(-1, 3)
        rgb_axis = pallete[None].repeat(100, 0).reshape(-1, 3)
        xyz_vis = np.concatenate([xyz, xyz_center, xyz_axis])
        rgb_vis = np.concatenate([color, rgb_center, rgb_axis])
        visualize_point_cloud(xyz_vis, rgb_vis)

In [ ]:
data_dir = "../data/videoartgs/sapien"
data_dir = "../data/videoartgs/realscan"
scenes = sorted(os.listdir(data_dir))
scene_names = [os.path.basename(s) for s in scenes if os.path.isdir(os.path.join(data_dir, s))]
print("'"+"' '".join(scene_names)+"'")
scene_names = ['30666_new']
scene_names = ['box_4r', 'cabinet_2r_4p', 'coffeemachine_2r']
scene_names = ['cabinet_2r_2p']
scene_names = ['microwave_1r', 'mac_1r', 'chair_1r', 'coffeemachine_2r']
for scene_name in scene_names:
    # print(scene_name)
    vis_init_cano(data_dir, scene_name)

'box_4r' 'cab1' 'cab_1r_1p' 'cab_2r' 'cabinet_2r_2p' 'cabinet_2r_4p' 'cabinet_white_2r' 'chair_1r' 'coffeemachine_1r_1p' 'coffeemachine_2r' 'faucet_2r' 'floor13' 'fridge' 'fridge_2r_2p' 'light' 'mac_1r' 'microwave1' 'microwave_1r' 'microwave_thirdview' 'microwave_w' 'microwave_white' 'office' 'printer_1r_2p' 'printer_2r_1p' 'real_2r1p' 'robocab' 'scene1' 'scene_2r1p' 'scene_2r1p (copy)' 'storage_1r' 't_1p' 'table_1p'
[0.16536264 0.13778497 0.09264243] [-0.22007325 -0.13933696 -0.09208003]
[0.19254997 0.27372226 0.1177682 ] [-0.16293025 -0.31920388 -0.07487834]
[0.31417948 0.2793337  0.23601678] [-0.23647663 -0.1755074  -0.20710267]
[0.26863068 0.2280355  0.25525534] [-0.3635419  -0.26654378 -0.16046728]
